In [1]:
# Cell 1: Core Utilities for COMPASS
# - Modular arithmetic and ring operations
# - Discrete Gaussian sampling
# - Partial Fourier (Vandermonde) evaluation
# - Compact witness serialization

import numpy as np, math, hashlib
from dataclasses import dataclass
from typing import List, Tuple

# ------------------ Utilities ------------------
def mod_q(x, q: int) -> np.ndarray:
    return (np.asarray(x, dtype=np.int64) % q).astype(np.int64)

def zero_center(vec: np.ndarray, q: int) -> np.ndarray:
    return ((vec + q//2) % q) - q//2

def negacyclic_conv(a: np.ndarray, b: np.ndarray, q: int) -> np.ndarray:
    N = a.shape[0]
    full = np.convolve(a.astype(np.int64), b.astype(np.int64))
    res = np.zeros(N, dtype=np.int64)
    res[:N] = full[:N]
    tail = full[N:]
    for j, val in enumerate(tail):
        res[j % N] -= val
    return mod_q(res, q)

def shake256(data: bytes, outlen: int) -> bytes:
    return hashlib.shake_256(data).digest(outlen)

def bytes_per_q(q: int) -> int:
    return ((q - 1).bit_length() + 7) // 8

def vec_modq_to_bytes(vec: np.ndarray, q: int) -> bytes:
    b = bytes_per_q(q)
    return b"".join(int(int(x) % q).to_bytes(b, "big") for x in vec)

def vec_modq_from_bytes(data: bytes, n: int, q: int) -> Tuple[np.ndarray, int]:
    b = bytes_per_q(q)
    need = n * b
    if len(data) < need:
        raise ValueError(f"need {need} bytes, got {len(data)}")
    chunk = data[:need]
    arr = np.frombuffer(chunk, dtype=np.uint8).reshape(-1, b)
    vals = np.array([int.from_bytes(row.tobytes(), "big") for row in arr], dtype=np.int64)
    return vals, need

# ------------------ Compact PASS-style c encoding ------------------
# Format: 8 bytes of sign bits (bit i = 1 means -1, 0 means +1), then κ indices (uint16 BE)
# Assumptions: κ <= 64, N <= 65535
def pack_c(c: np.ndarray, kappa: int) -> bytes:
    nz = np.flatnonzero(c)
    assert len(nz) == kappa, "c must have exactly κ nonzeros"
    signs = [(0 if c[i] == 1 else 1) for i in nz]  # 0=>+1, 1=>-1
    sign_bits = 0
    for i, bit in enumerate(signs[:64]):
        sign_bits |= (bit & 1) << i
    idx_bytes = b"".join(int(i).to_bytes(2, "big") for i in nz)
    return sign_bits.to_bytes(8, "big") + idx_bytes

def unpack_c(data: bytes, N: int, kappa: int) -> Tuple[np.ndarray, int]:
    sign_bits = int.from_bytes(data[:8], "big")
    idx_bytes = data[8:8 + 2*kappa]
    nz_idx = [int.from_bytes(idx_bytes[i:i+2], "big") for i in range(0, 2*kappa, 2)]
    c = np.zeros(N, dtype=np.int64)
    for i, idx in enumerate(nz_idx):
        sign = -1 if ((sign_bits >> i) & 1) else 1
        c[idx] = sign
    return c, (8 + 2*kappa)

# ------------------ Partial Fourier (FΩ) ------------------
def find_primitive_root(q: int) -> int:
    for g in range(2, q-1):
        if pow(g, (q-1)//2, q) != 1:
            return g
    return 3

def find_g_2N(q: int, N: int) -> int:
    r = find_primitive_root(q)
    g = pow(r, (q-1)//(2*N), q)
    if pow(g, N, q) != q-1:
        for k in range(2, 2*N):
            cand = pow(r, ((q-1)//(2*N))*k, q)
            if pow(cand, N, q) == q-1:
                return cand
    return g

class PartialFourier:
    """
    Partial Fourier Transform with overflow protection for large q.

    Evaluates polynomials at subset Ω of roots using vectorized operations
    with uint64 arithmetic and periodic modular reduction.
    """

    def __init__(self, q: int, N: int, t: int, Omega=None, g=None):
        """
        Initialize Partial Fourier transform.

        Args:
            q: Modulus (must satisfy q ≡ 1 mod 2N)
            N: Ring dimension (power of 2)
            t: Number of evaluation points (|Ω|)
            Omega: Optional preset exponents (odd powers)
            g: Optional preset primitive 2N-th root
        """
        self.q, self.N, self.t = q, N, t
        self.g = g if g is not None else find_g_2N(q, N)

        # Choose Omega (odd exponents for roots of x^N + 1)
        if Omega is None:
            odd = np.arange(1, 2*N, 2, dtype=np.int64)
            rng = np.random.default_rng()
            self.Omega = np.sort(rng.choice(odd, size=t, replace=False))
        else:
            self.Omega = np.array(Omega, dtype=np.int64)
            assert np.all(Omega % 2 == 1), "Omega must contain odd exponents"
            self.Omega = np.sort(Omega)

        # Precompute Vandermonde matrix as uint64 for overflow safety
        self.P = np.empty((t, N), dtype=np.uint64)
        for j, r in enumerate(self.Omega):
            root = pow(self.g, int(r), self.q)
            power = 1
            for i in range(N):
                self.P[j, i] = power
                power = (power * root) % self.q

        # Determine evaluation strategy based on q
        if self.q > (1 << 29):  # ~500M threshold
            # For very large q (close to 2^32), be extra conservative
            if self.q > (1 << 31):  # q > 2^31
                # Can only accumulate 1 term at a time!
                self.reduce_every = 1
                print(f"  Using ultra-safe mode: q={q} requires reduce_every=1")
            else:
                # For large q, compute safe reduction frequency
                # uint64 can hold up to 2^64
                # Each term is at most q * q ≈ 2^60 for q ≈ 2^30
                # Safely accumulate multiple terms before reduction
                max_terms = (1 << 63) // (self.q * self.q)  # Conservative estimate
                self.reduce_every = max(1, max_terms // 2)
            self.use_safe_mode = True
            print(f"  Precomputed Vandermonde matrix: {self.P.shape} ({self.P.nbytes / 1024:.1f} KB)")
            print(f"  Using overflow-safe mode: reduce every {self.reduce_every} terms")
        else:
            self.reduce_every = self.N
            self.use_safe_mode = False
            print(f"  Precomputed Vandermonde matrix: {self.P.shape} ({self.P.nbytes / 1024:.1f} KB)")

    def F(self, f: np.ndarray) -> np.ndarray:
        """
        Evaluate polynomial f at roots in Ω.

        Computes f̂|Ω = P·f (mod q) with overflow protection for large q.

        Args:
            f: Coefficient vector (length N)

        Returns:
            Evaluation vector (length t): [f(ω^r1), ..., f(ω^rt)]

        Time: O(t·N) with vectorized NumPy operations
        """
        # Convert to uint64 and reduce modulo q
        f_mod = (f % self.q).astype(np.uint64)

        if not self.use_safe_mode:
            # Fast path for small q: single matmul
            result = (self.P @ f_mod) % self.q
            return result.astype(np.int64)

        # Safe path for large q: chunked matmul with periodic reduction
        result = np.zeros(self.t, dtype=np.uint64)

        for start in range(0, self.N, self.reduce_every):
            end = min(start + self.reduce_every, self.N)

            # Compute partial dot product (won't overflow due to small chunk)
            chunk_result = self.P[:, start:end] @ f_mod[start:end]

            # Accumulate with modular reduction
            result = (result + chunk_result) % self.q

        return result.astype(np.int64)

# ------------------ Witness serialization (compact c + mod-q vectors) ------------------
@dataclass
class WitnessPacked:
    """Packed witness: (Z_i, z_i, c1, c2) with compact encoding"""
    data: bytes

def witness_serialize(Zi: np.ndarray, zi: np.ndarray, c1: np.ndarray, c2: np.ndarray,
                      N: int, q: int, kappa: int) -> WitnessPacked:
    """
    Serialize witness components into compact format.

    Args:
        Zi, zi: mod-q vectors (length N)
        c1, c2: ternary challenge vectors with exactly κ nonzeros
        N: ring dimension
        q: modulus
        kappa: challenge sparsity
    """
    out  = vec_modq_to_bytes(Zi, q)
    out += vec_modq_to_bytes(zi, q)
    out += pack_c(c1, kappa)
    out += pack_c(c2, kappa)
    return WitnessPacked(out)

def witness_deserialize(buf: bytes, N: int, q: int, kappa: int) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Deserialize witness from compact format.

    Returns:
        (Zi, zi, c1, c2) tuple
    """
    off = 0
    Zi, used = vec_modq_from_bytes(buf[off:], N, q); off += used
    zi, used = vec_modq_from_bytes(buf[off:], N, q); off += used
    c1, used = unpack_c(buf[off:], N, kappa); off += used
    c2, used = unpack_c(buf[off:], N, kappa); off += used
    return (Zi % q, zi % q, c1, c2)


In [2]:
# Cell 2: Hash & Domain Separation Functions
def Hash_C(data: bytes, outlen: int = 64) -> bytes:
    """Challenge hash: arbitrary data -> τ bits for FormatC"""
    return shake256(data, outlen)

def FormatC(seed: bytes, N: int, kappa: int) -> np.ndarray:
    """Deterministic sparse ternary: exactly κ nonzeros from {-1,+1}"""
    rng = np.random.default_rng(int.from_bytes(seed, 'big'))
    idx = rng.choice(N, size=kappa, replace=False)
    sgn = rng.choice([-1, 1], size=kappa)
    c = np.zeros(N, dtype=np.int64)
    c[idx] = sgn
    return c

def Hash_beta(c1: np.ndarray, c2: np.ndarray, fhat: np.ndarray, q: int) -> int:
    """Aggregation weight: ±1 based on challenge/element hash"""
    data = vec_modq_to_bytes(c1, q) + vec_modq_to_bytes(c2, q) + vec_modq_to_bytes(fhat % q, q)
    return 1 if (shake256(data, 1)[0] & 1) else -1

def Hash_Acc(Z_hat: np.ndarray, q: int, m: int) -> np.ndarray:
    """Accumulator digest: Z_hat -> m elements in Z_q"""
    need = 2 * m
    raw = shake256(vec_modq_to_bytes(Z_hat % q, q), need)
    vals = np.frombuffer(raw, dtype=np.uint8).astype(np.int64)
    limbs16 = (vals[0::2] << 8) + vals[1::2]
    return (limbs16 % q)[:m]

def HashToLat(cert: bytes, N: int) -> np.ndarray:
    """
    Map certificate to short ring vector in B_∞(1).

    Implementation: HashToLat = Φ ∘ HashC where:
    - HashC: certificate -> raw bytes via SHAKE256
    - Φ: deterministic rounding/centering to {-1, 0, +1}^N

    This is preprocessing done BEFORE Accumulate.

    Args:
        cert: Certificate bytes (email, ID, etc.)
        N: Ring dimension

    Returns:
        f ∈ B_∞(1): short vector with coefficients in {-1, 0, +1}
    """
    raw = shake256(cert, N)

    # Deterministic map Φ: bytes -> {-1, 0, 1}^N
    # Use mod 3 to get uniform distribution over {0, 1, 2}, then shift
    f = np.zeros(N, dtype=np.int64)
    for i in range(N):
        val = raw[i] % 3  # {0, 1, 2}
        f[i] = val - 1     # {-1, 0, +1}

    return f

In [3]:
# Cell 3: COMPASS Accumulator Class

class COMPASS:
    """
    COMPASS: Compact PASS-lineage Accumulator with Succinct Proofs
    """
    def __init__(self, K=20, param_set="set1", lambda_sec=128):
        """
        Initialize COMPASS parameters (but not keys).

        This sets up the mathematical structure but doesn't generate secrets yet.
        Setup() must be called separately to generate keys.
        """
        # Choose parameter set
        if param_set == "set1":
            self.N = 512
            self.q = 205207553
            self.kappa = 44
            self.sigma = 11336
            self.t = 256

        elif param_set == "set2":
            self.N = 1024
            self.q = 4294957057
            self.kappa = 36
            self.sigma = 167771
            self.t = 512

        self.m = 128    # HashAcc output dimension
        self.tau = 512  # HashC output bits
        self.lambda_sec = lambda_sec

        # Derived parameters
        self.sy = self._compute_gaussian_threshold()
        self.Boundz = self._compute_Boundz()
        self.M = self._compute_M()
        self.BoundZ = None
        self.K = K
        self.ctx = None
        # Compute BoundZ
        self.BoundZ = self._compute_BoundZ_for_K(K)
        print(f"  BoundZ for K={K}: {self.BoundZ:.2f}")

        # Initialize PartialFourier (generates Ω)
        self.PF = PartialFourier(self.q, self.N, self.t)
        assert pow(self.PF.g, self.N, self.q) == self.q - 1, "g^N must be -1 mod q"

        # Keys and oracles set to None (will be filled by Setup)
        self.sk = None
        self.pk = None
        self.Hc1 = None  # Domain-separated challenge hash
        self.Hc2 = None  # Domain-separated challenge hash
        self.Hbeta = None  # Aggregation weight hash
        self.HAcc = None  # Accumulator hash

        print(f"COMPASS initialized with {param_set}: N={self.N}, q={self.q}, κ={self.kappa}, t={self.t}")

    def Setup(self, ctx=None, epoch="2025-01"):
        """
        Setup(1^λ) → (pp, sk)

        Generate manager's keypair and define domain-separated hash oracles.

        Args:
            ctx: Context string for domain separation (scheme ID, epoch, etc.)
            epoch: Epoch/version string for domain separation

        Returns:
            pp: Public parameters (dict)
            sk: Secret key (stored internally as self.sk)
        """
        print(f"Running Setup with security parameter λ={self.lambda_sec}...")

        # 1. Sample secret key sk ← B_∞(1)
        self.sk = np.random.choice([-1, 0, 1], size=self.N).astype(np.int64)

        # 2. Compute public key pk = F_Ω(sk)
        self.pk = self.PF.F(self.sk)

        # 3. Define domain-separated hash oracles
        if ctx is None:
            # Hash of parameters for uniqueness
            params_str = f"N={self.N}||q={self.q}||t={self.t}||kappa={self.kappa}||sigma={self.sigma}"
            params_hash = shake256(params_str.encode(), 16).hex()
            ctx = f"COMPASS-v1||{params_hash}||{epoch}"

        self.ctx = ctx.encode()
        print(f"  Context: {ctx[:60]}...")

        # Store as methods with fixed domain separation
        # These capture self and ctx in closure
        def _Hc1(y_hat_omega, f_hat_omega):
            """Challenge hash c1: binds commitment to member vector"""
            data = self.ctx + b"||C1||" + vec_modq_to_bytes(y_hat_omega, self.q) + vec_modq_to_bytes(f_hat_omega, self.q)
            seed = Hash_C(data, 64)
            return FormatC(seed, self.N, self.kappa)

        def _Hc2(y_hat_omega, pk):
            """Challenge hash c2: binds commitment to public key"""
            data = self.ctx + b"||C2||" + vec_modq_to_bytes(y_hat_omega, self.q) + vec_modq_to_bytes(pk, self.q)
            seed = Hash_C(data, 64)
            return FormatC(seed, self.N, self.kappa)

        def _Hbeta(c1, c2, f_hat_omega):
            """Aggregation weight: ±1"""
            data = self.ctx + b"||BETA||"
            return Hash_beta(c1, c2, f_hat_omega, self.q)

        def _HAcc(Z_hat_omega):
            """Accumulator digest"""
            return Hash_Acc(Z_hat_omega, self.q, self.m)

        # Assign to instance
        self.Hc1 = _Hc1
        self.Hc2 = _Hc2
        self.Hbeta = _Hbeta
        self.HAcc = _HAcc

        # Build public parameters
        pp = {
            'N': self.N,
            'q': self.q,
            'g': self.PF.g,
            'sigma': self.sigma,
            'Omega': self.PF.Omega,
            't': self.t,
            'kappa': self.kappa,
            'm': self.m,
            'tau': self.tau,
            'pk': self.pk,
            'ctx': self.ctx,
            'Boundz': self.Boundz
        }

        print(f"✓ Setup complete. pk has dimension {len(self.pk)}")
        return pp, self.sk

    def Accumulate(self, F: List[np.ndarray]):
        """
        Accumulate(pp, sk, K, {f_i}) → (Acc, Z, L)

        Main accumulation algorithm with rejection sampling.
        This is the most computation-intensive part.

        Args:
            F: List of member vectors f_i ∈ B_∞(1) (NOT certificates!)
               Use HashToLat(cert, N) externally to convert certificates to f_i

        Returns:
            Acc: Accumulator value (m-dimensional vector in Z_q)
            Z: Aggregate state (polynomial in Rq)
            L: List of tuples (β_i, c_i1, c_i2, z_i) for witness extraction
        """
        if self.sk is None:
            raise RuntimeError("Must call Setup() before Accumulate()")

        K = len(F)
        #self.K = K
        #print(f"\nAccumulating {K} members...")
        if K > self.K:
            print(f"Entered member number exceeded set K, this will be unsafe.")

        F_hat = [self.PF.F(f) for f in F]  # Precompute evaluations

        # Compute BoundZ
        #self.BoundZ = self._compute_BoundZ_for_K(K)
        #print(f"  BoundZ for K={K}: {self.BoundZ:.2f}")

        outer_attempts = 0
        max_outer_attempts = 10
        while outer_attempts < max_outer_attempts:
            sum_attempts = 0
            outer_attempts += 1
            Z = np.zeros(self.N, dtype=np.int64)
            L = []
            for i in range(K):
                # Inner loop: rejection sampling for single member
                inner_attempts = 0
                max_inner_attempts = 30000
                while inner_attempts < max_inner_attempts:
                    inner_attempts += 1

                    # 1. Sample commitment y ← D^N_σ
                    y = np.random.normal(0.0, self.sigma, size=self.N).round().astype(np.int64)
                    if np.max(np.abs(y)) > self.sy:
                        continue  # Resample if clipping threshold exceeded
                    y_hat_omega = self.PF.F(y)

                    # 2. Compute dual challenges
                    c1 = self.Hc1(y_hat_omega, F_hat[i])
                    c2 = self.Hc2(y_hat_omega, self.pk)

                    # 3. Compute response z = sk * c1 + f * c2 + y
                    u = (negacyclic_conv(self.sk, c1, self.q) + negacyclic_conv(F[i], c2, self.q)) % self.q
                    z = (u + y) % self.q
                    z = zero_center(z, self.q)  # Center for norm computation
                    u = zero_center(u, self.q)  # Center for rejection sampling

                    # 4. Rejection sampling checks
                    z_norm = np.linalg.norm(z)
                    if z_norm > self.Boundz:
                        continue  # Reject: z too large

                    # 5. Weighted rejection sampling
                    if not self._rejection_sampling(u, z):
                        continue

                    # Accepted!
                    sum_attempts += inner_attempts
                    if inner_attempts > 1:
                        print(f"  Member {i+1}: accepted after {inner_attempts} attempts")
                    break

                if inner_attempts >= max_inner_attempts:
                    raise RuntimeError(f"Failed to find valid z for member {i} after {max_inner_attempts} attempts")


                # 6. Compute aggregation weight β_i and update Z
                beta_i = self.Hbeta(c1, c2, F_hat[i])
                Z = (Z + beta_i * z) % self.q
                L.append((beta_i, c1, c2, z))

            print(f"Empirical p ≈ {K/sum_attempts:.3e} "
                  f"(avg attempts/member ≈ {sum_attempts/K:.1f})")
            # Check aggregate bound
            Z_centered = zero_center(Z, self.q)
            Z_norm = np.linalg.norm(Z_centered)
            if Z_norm <= self.BoundZ:
                print(f"✓ Accumulation successful after {outer_attempts} outer attempt(s)")
                break
            else:
                print(f"  Outer attempt {outer_attempts}: ||Z|| = {Z_norm:.1f} > {self.BoundZ:.1f}, restarting...")

        if outer_attempts >= max_outer_attempts:
            raise RuntimeError(f"Failed to satisfy BoundZ after {max_outer_attempts} attempts")

        # Compute accumulator value
        Z_hat_omega = self.PF.F(Z)
        Acc = self.HAcc(Z_hat_omega)

        print(f"  Accumulator size: {len(Acc)} elements")
        print(f"  ||Z||_2 = {Z_norm:.2f}")

        return Acc, Z, L

    def Witness(self, Z: np.ndarray, L: List, i: int) -> WitnessPacked:
        """
        Witness(pp, Z, L, i) → w_i

        Extract witness for member i. Lightweight operation.

        Args:
            Z: Aggregate state
            L: Auxiliary list from Accumulate
            i: Member index (0-indexed)

        Returns:
            w_i: Witness tuple (Z_i, z_i, c_i1, c_i2)
        """
        if i < 0 or i >= len(L):
            raise ValueError(f"Invalid member index {i}")

        beta_i, ci1, ci2, zi = L[i]

        # Compute Z_i = Z - β_i * z_i (mod q)
        Zi = (Z - beta_i * zi) % self.q

        return self.serialize_witness((Zi, zi, ci1, ci2))

    def Verify(self, Acc: np.ndarray, f: np.ndarray, witness_packed: WitnessPacked) -> bool:
        """
        Verify(pp, Acc, f, w) → {0, 1}

        Verify membership witness.

        Args:
            Acc: Accumulator value
            f: Member vector
            witness: Tuple (Z_i, z_i, c_i1, c_i2)

        Returns:
            True if valid, False otherwise
        """
        if self.BoundZ is None:
            raise RuntimeError("BoundZ not set - must Accumulate before Verify")
        Zi, zi, ci1, ci2 = self.deserialize_witness(witness_packed.data)

        # 1. Norm check on z_i
        zi_centered = zero_center(zi, self.q)
        if np.linalg.norm(zi_centered) > self.Boundz:
            return False

        # 2. Compute evaluations (all return uint64 internally, cast to int64)
        zi_hat = self.PF.F(zi).astype(np.uint64)
        ci1_hat = self.PF.F(ci1).astype(np.uint64)
        ci2_hat = self.PF.F(ci2).astype(np.uint64)
        f_hat = self.PF.F(f).astype(np.uint64)
        pk_uint = self.pk.astype(np.uint64)

        # 3. Reconstruct y'_hat = z_hat - pk ⊙ c1_hat - f_hat ⊙ c2_hat
        #y_prime_hat = (zi_hat - self.pk * ci1_hat - f_hat * ci2_hat) % self.q
        # Compute each multiplication separately with immediate modulo
        term1 = (pk_uint * ci1_hat) % self.q
        term2 = (f_hat * ci2_hat) % self.q
        # Subtraction in modular arithmetic (handle underflow)
        y_prime_hat = (zi_hat + self.q - term1 + self.q - term2) % self.q
        y_prime_hat = y_prime_hat.astype(np.int64)

        # 4. Recompute challenges
        ci1_prime = self.Hc1(y_prime_hat, f_hat)
        ci2_prime = self.Hc2(y_prime_hat, self.pk)

        # 5. Challenge consistency check
        if not (np.array_equal(ci1_prime, ci1) and np.array_equal(ci2_prime, ci2)):
            return False

        # 6. Reconstruct aggregate Z' = Z_i + β_i * z_i
        beta_i = self.Hbeta(ci1, ci2, f_hat)
        Z_prime = (Zi + beta_i * zi) % self.q

        # 7. Aggregate norm check
        Z_prime_centered = zero_center(Z_prime, self.q)
        if np.linalg.norm(Z_prime_centered) > self.BoundZ:
            return False

        # 8. Recompute accumulator
        Z_prime_hat = self.PF.F(Z_prime)
        Acc_prime = self.HAcc(Z_prime_hat)

        # 9. Final check
        return np.array_equal(Acc_prime, Acc)

    def serialize_witness(self, wit):
        """Convenience wrapper for witness serialization"""
        return witness_serialize(*wit, self.N, self.q, self.kappa)

    def deserialize_witness(self, buf):
        """Convenience wrapper for witness deserialization"""
        return witness_deserialize(buf, self.N, self.q, self.kappa)

    def _compute_M(self):
        """
        Compute rejection sampling constant M.
        """
        U = 2 * self.kappa * np.sqrt(self.N)
        exponent = (U**2 + 2 * U * self.Boundz) / (2 * self.sigma**2)
        return np.exp(exponent)

    def _compute_BoundZ_for_K(self, K: int) -> float:
        """
        Compute aggregate bound BoundZ for K members.
        """
        return 2 * self.sigma * np.sqrt(K * self.N)

    def _compute_gaussian_threshold(self):
        """Compute clipping threshold s_y for Gaussian sampling"""
        return self.sigma * np.sqrt(2 * (self.lambda_sec * np.log(2) + np.log(2 * self.N)))

    def _compute_Boundz(self):
        """Compute L2 bound on z for verification"""
        # From paper: ||z||_2 ≤ 2σ√N
        return 2 * self.sigma * np.sqrt(self.N)

    def _rejection_sampling(self, u: np.ndarray, z: np.ndarray) -> bool:
        """
        Weighted rejection sampling (Lyubashevsky-style).

        Accept z with probability:
        P_accept = min(1, (exp(-2⟨z,u⟩ + ||u||²) / (2σ²))/M))

        From Equation (1) in paper.
        """
        # Compute exponent: (-2⟨z,u⟩ + ||u||²₂) / (2σ²)
        inner_product = np.dot(z, u)
        u_norm_sq = np.dot(u, u)

        exponent = (-2 * inner_product + u_norm_sq) / (2 * self.sigma**2)

        # P_accept = min(1, exp(exponent) / M)
        prob = min(1.0, np.exp(exponent) / self.M)

        # Sample uniform and accept if below probability
        return np.random.random() < prob

In [4]:
# Cell 4: Test COMPASS with Timing Analysis

import time

def test_compass():
    """Test COMPASS with proper HashToLat usage and performance tracking"""
    print("="*60)
    print("TESTING CORRECTED COMPASS")
    print("="*60)

    # 1. Initialize
    compass = COMPASS(K=100000, param_set="set1", lambda_sec=128)

    # 2. Setup
    pp, sk = compass.Setup(ctx=None, epoch="2025-01")
    print(f"\n✓ Setup complete")

    # 3. Create certificates and convert to lattice vectors
    K = 200
    certificates = [f"user_{i}@example.com".encode() for i in range(K)]

    # IMPORTANT: Use HashToLat BEFORE calling Accumulate
    F = [HashToLat(cert, compass.N) for cert in certificates]
    print(f"\n✓ Converted {K} certificates to lattice vectors")

    # 4. Accumulate with timing
    print(f"\n{'='*60}")
    print(f"ACCUMULATE (K={K} members)")
    print(f"{'='*60}")
    start = time.time()
    Acc, Z, L = compass.Accumulate(F)
    time_accumulate = time.time() - start

    print(f"\n⏱️  Total Accumulate time: {time_accumulate:.3f}s ({time_accumulate*1000:.1f}ms)")
    print(f"   Average per member: {time_accumulate/K:.3f}s ({time_accumulate*1000/K:.1f}ms)")

    # 5. Generate witnesses (just show size for first few)
    print(f"\n{'='*60}")
    print(f"WITNESS GENERATION")
    print(f"{'='*60}")
    for i in range(min(5, K)):
        w_packed = compass.Witness(Z, L, i)
        print(f"  Witness {i}: {len(w_packed.data)} bytes")
    if K > 5:
        print(f"  ... ({K-5} more witnesses)")

    # 6. Verify with timing
    print(f"\n{'='*60}")
    print(f"VERIFICATION (K={K} members)")
    print(f"{'='*60}")

    verification_times = []
    all_valid = True

    for i in range(K):
        w_packed = compass.Witness(Z, L, i)

        start = time.time()
        valid = compass.Verify(Acc, F[i], w_packed)
        verify_time = time.time() - start
        verification_times.append(verify_time)

        status = "✓ VALID" if valid else "✗ INVALID"
        if i < 5 or not valid:  # Show first 5 and any failures
            print(f"  Member {i}: {status} ({verify_time*1000:.2f}ms)")

        if not valid:
            print(f"    ERROR: Verification failed!")
            all_valid = False

    if K > 5 and all_valid:
        print(f"  ... ({K-5} more members verified)")

    # Verification statistics
    avg_verify = np.mean(verification_times)
    min_verify = np.min(verification_times)
    max_verify = np.max(verification_times)

    print(f"\n⏱️  Verification times:")
    print(f"   Average: {avg_verify*1000:.2f}ms")
    print(f"   Min: {min_verify*1000:.2f}ms")
    print(f"   Max: {max_verify*1000:.2f}ms")
    print(f"   Total: {sum(verification_times):.3f}s")

    if not all_valid:
        return False

    # 7. Negative test
    print(f"\n{'='*60}")
    print(f"NEGATIVE TEST (Forgery Attempt)")
    print(f"{'='*60}")
    fake_cert = b"attacker@evil.com"
    fake_f = HashToLat(fake_cert, compass.N)
    w0_packed = compass.Witness(Z, L, 0)

    start = time.time()
    forged = compass.Verify(Acc, fake_f, w0_packed)
    forgery_time = time.time() - start

    if forged:
        print(f"  ✗ ERROR: Forgery succeeded!")
        return False
    else:
        print(f"  ✓ Forgery rejected ({forgery_time*1000:.2f}ms)")

    # Final summary
    print(f"\n{'='*60}")
    print(f"PERFORMANCE SUMMARY")
    print(f"{'='*60}")
    print(f"Parameter set: {compass.N=}, {compass.q=}, {compass.kappa=}, {compass.t=}")
    print(f"Members (K): {K}")
    print(f"")
    print(f"Accumulate:")
    print(f"  Total time: {time_accumulate:.3f}s")
    print(f"  Per member: {time_accumulate*1000/K:.1f}ms")
    print(f"")
    print(f"Witness size: {len(compass.Witness(Z, L, 0).data)} bytes")
    print(f"")
    print(f"Verify:")
    print(f"  Average time: {avg_verify*1000:.2f}ms")
    print(f"  Throughput: {1/avg_verify:.1f} verifications/sec")
    print(f"")
    print(f"Accumulator size: {len(Acc)} elements × {bytes_per_q(compass.q)} bytes = {len(Acc) * bytes_per_q(compass.q)} bytes")

    print(f"\n{'='*60}")
    print("ALL TESTS PASSED! ✓")
    print(f"{'='*60}")
    return True

# Run the test
test_compass()

TESTING CORRECTED COMPASS
  BoundZ for K=100000: 162227626.19
  Precomputed Vandermonde matrix: (256, 512) (1024.0 KB)
COMPASS initialized with set1: N=512, q=205207553, κ=44, t=256
Running Setup with security parameter λ=128...
  Context: COMPASS-v1||7ecc84cf6200dd0111448445a85a1de0||2025-01...
✓ Setup complete. pk has dimension 256

✓ Setup complete

✓ Converted 200 certificates to lattice vectors

ACCUMULATE (K=200 members)
  Member 1: accepted after 4179 attempts
  Member 2: accepted after 4251 attempts
  Member 3: accepted after 5390 attempts
  Member 4: accepted after 3647 attempts
  Member 5: accepted after 3154 attempts
  Member 6: accepted after 1040 attempts
  Member 7: accepted after 436 attempts
  Member 8: accepted after 469 attempts
  Member 9: accepted after 6128 attempts
  Member 10: accepted after 843 attempts
  Member 11: accepted after 1009 attempts
  Member 12: accepted after 392 attempts
  Member 13: accepted after 1321 attempts
  Member 14: accepted after 5045 att

True